
This notebook is adapted from [Dataflowr Module's 9c online ressource](https://dataflowr.github.io/website/modules/9c-flows/#density_estimation_using_real_nvp).

To access this notebook on colab: https://colab.research.google.com/drive/1uXfS31e0ttipGyaKlZRyVasTtb-VqxpZ?usp=sharing.

# 1. Normalizing flows

The image below is taken from this very good blog post on normalizing flows : [blogpost](https://lilianweng.github.io/lil-log/2018/10/13/flow-based-deep-generative-models.html)

![](https://raw.githubusercontent.com/dataflowr/website/master/modules/extras/flows/three-generative-models.png)

## Principles
A **flow-based generative model** is constructed by a sequence of **invertible** transformations. The main advantage of flows is that the model explicitly learns the data distribution $p(\mathbf{x})$ and therefore the loss function is simply the negative log-likelihood.

Given a sample $\mathbf{x}$ and a prior $p(\mathbf{z})$, we compute $f(\mathbf{x}) = \mathbf{z}$ with an invertible function $f$ that will be learned. Given $f$ and the prior $p(\mathbf{z})$, we can compute the evidence $p(\mathbf{x})$ thanks to the change of variable formula:
$$
\begin{align*}
\mathbf{z} &\sim p(\mathbf{z}), \mathbf{z} = f(\mathbf{x}), \\
p(\mathbf{x}) 
&= p(\mathbf{z}) \left\vert \det \dfrac{d \mathbf{z}}{d \mathbf{x}} \right\vert  
= p(f(\mathbf{x})) \left\vert \det \dfrac{\partial f(\mathbf{x})}{\partial \mathbf{x}} \right\vert
\end{align*}
$$
where $\dfrac{\partial f(\mathbf{x})}{\partial \mathbf{x}}$ is the Jacobian matrix of $f$.
Recall that given a function mapping a $n$-dimensional input vector $\mathbf{x}$ to a $m$-dimensional output vector, $f: \mathbb{R}^n \mapsto \mathbb{R}^m$, the matrix of all first-order partial derivatives of this function is called the **Jacobian matrix**, $J_f$ where one entry on the i-th row and j-th column is $(J_f(\mathbf{x}))_{ij} = \frac{\partial f_i(\mathbf{x})}{\partial x_j}$:
$$
{J_f(\mathbf{x})} = \begin{bmatrix}
\frac{\partial f_1(\mathbf{x})}{\partial x_1} & \dots & \frac{\partial f_1(\mathbf{x})}{\partial x_n} \\[6pt]
\vdots & \ddots & \vdots \\[6pt]
\frac{\partial f_m(\mathbf{x})}{\partial x_1} & \dots & \frac{\partial f_m(\mathbf{x})}{\partial x_n} \\[6pt]
\end{bmatrix}
$$
Below, we will parametrize $f$ with a neural network and learn $f$ by maximizing $\ln p(\mathbf{x})$. More precisely, given a dataset $(\mathbf{x}_1,\dots,\mathbf{x}_n)$ and a model provided by a prior $p(\mathbf{z})$ and a neural network $f$, we optimize the weights of $f$ by minimizing:
$$
-\sum_{i}\ln p(\mathbf{x_i}) = \sum_i -\ln p(f(\mathbf{x}_i)) -\ln\left\vert \det \dfrac{\partial f(\mathbf{x}_i)}{\partial \mathbf{x}} \right\vert.
$$

**We need to ensure that $f$ is always invertible and that the determinant is simple to compute.**

## [Density estimation using Real NVP](https://arxiv.org/abs/1605.08803) 
by Laurent Dinh, Jascha Sohl-Dickstein, Samy Bengio (2016)

[Real NVP](https://arxiv.org/abs/1605.08803) uses function $f$ obtained by stacking affine coupling layers which for an input $\mathbf{x}\in \mathbb{R}^D$ produce the output $\mathbf{y}\in\mathbb{R}^D$ defined by ($d<D$): 
$$
\begin{align}
\label{eq:aff}
\mathbf{y}_{1:d} &= \mathbf{x}_{1:d}\\
\mathbf{y}_{d+1:D} &= \mathbf{x}_{d+1:D} \odot \exp\left(s(\mathbf{x}_{1:d})\right) +t(\mathbf{x}_{1:d}) ,
\end{align}
$$
where $s$ (scale) and $t$ (translation) are neural networks mapping $\mathbb{R}^d$ to $\mathbb{R}^{D-d}$ and $\odot$ is the element-wise product.

> **Q1.** For any functions $s$ and $t$, the affine coupling layer is invertible. Write down the inversion equations.

> **Q2.** Write the associated Jacobian matrix and its determinant. Hint: $J$ is a lower triangular matrix.

### Practical detail about the coupling layer
In one affine coupling layer, some dimensions (channels) remain unchanged. To make sure all the inputs have a chance to be altered, the model reverses the ordering in each layer so that different components are left unchanged. Following such an alternating pattern, the set of units which remain identical in one transformation layer are always modified in the next. 

This can be implemented with binary masks. First, we can extend the scale and neural networks to mappings form $\mathbb{R}^D$ to $\mathbb{R}^D$. Then taking a mask $\mathbf{b} = (1,\dots,1,0,\dots,0)$ with $d$ ones, so that we have for the affine layer:
\begin{align*}
\mathbf{y} = \mathbf{x} \odot \exp\big((1-\mathbf{b}) \odot s(\mathbf{b} \odot \mathbf{x})\big) + (1-\mathbf{b}) \odot t(\mathbf{b} \odot \mathbf{x}).
\end{align*}
Note that we have
\begin{align*}
\ln \left\vert\det(J(\mathbf{x}))\right\vert = \sum_{j=1}^{D} \Big((1-\mathbf{b})\odot s(\mathbf{b} \odot \mathbf{x})\Big)_j,
\end{align*}
and to invert the affine layer:
\begin{align*}
\mathbf{x} = \left( \mathbf{y} -(1-\mathbf{b}) \odot t(\mathbf{b} \odot \mathbf{y})\right)\odot \exp\left( -(1-\mathbf{b}) \odot s(\mathbf{b} \odot \mathbf{y})\right)
\end{align*}
Now we alternates the binary mask $\mathbf{b}$ from one coupling layer to the other. 

Note, that the formula given in the paper is sligthly different:
$$\mathbf{y} = \mathbf{b} \odot \mathbf{x} + (1 - \mathbf{b}) \odot \Big(\mathbf{x} \odot \exp\big(s(\mathbf{b} \odot \mathbf{x})\big) + t(\mathbf{b} \odot \mathbf{x})\Big),$$
but the 2 formulas give the same result!

# 2. Implementation of Real NVP

In [ ]:
# Standard torch, sklearn and matplotlib imports

%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from tqdm import tqdm

from pylab import rcParams
rcParams['figure.figsize'] = 5, 4
rcParams['figure.dpi'] = 150

import torch
from torch import nn
from torch import distributions
from torch.nn.parameter import Parameter

from sklearn import datasets

# Formatting the plots
plt.rcParams['figure.figsize'] = [6,6]
plt.rcParams['font.size'] = 18
plt.rcParams['font.weight'] = 'normal'
plt.style.use('default')
mpl.rcParams['mathtext.fontset'] = 'cm'
mpl.rcParams['mathtext.rm'] = 'serif'
mpl.rcParams['font.size'] = 22
mpl.rcParams['axes.formatter.limits'] = (-6, 6)
mpl.rcParams['axes.formatter.use_mathtext'] = True
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['mathtext.rm'] = 'Bitstream Vera Sans'
mpl.rcParams['mathtext.it'] = 'Bitstream Vera Sans:italic'
mpl.rcParams['mathtext.bf'] = 'Bitstream Vera Sans:bold'
mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True

We will make use of the samples given by the $d=2$ [two moons](TODO) dataset that we sample and plot in the cell below.

In [ ]:
noise = 0.05
noisy_moons = datasets.make_moons(n_samples=1000, noise=noise)[0].astype(np.float32)

fig, ax = plt.subplots(figsize=(5,5))
x = datasets.make_moons(n_samples=1000, noise=.05)[0].astype(np.float32)
plt.scatter(x[:, 0], x[:, 1], c='k', s=5)
plt.title(r'$X \sim p(X)$')

> **Q3.** Create two networks using the [`Sequential`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html) Torch class; one for $s$ and one for $t$. Both will consist of a cascade of 3 linear layers with LeakyReLU activation functions. The size of the hidden layers are fixed to $h=256$. For $s$, we'll add an additional [`Tanh`](https://docs.pytorch.org/docs/stable/generated/torch.tanh.html) activation function at the end to normalize it.

In [ ]:
# here the scaling and translation neural networks are defined:
nets = lambda: nn.Sequential(# YOUR CODE to define the neural network)
nett = lambda: nn.Sequential(# YOUR CODE to define the neural network)

Below we define the several masks alternance $\mathbf{b}$ for a model with 6 coupling layers.

In [ ]:
masks = torch.from_numpy(np.array([[0, 1], [1, 0], [0, 1], [1, 0], [0, 1], [1, 0]]).astype(np.float32))
# torch.Tensor of size number_of_coupling_layers x dim(X)
masks.shape

> **Q4.** How should be the prior distribution? Implement it as an instance of a Torch [distribution](https://docs.pytorch.org/docs/stable/distributions.html).

In [ ]:
from torch import distributions
prior = # YOUR CODE to define the prior distribution

To compute the log probability and obtain sample from a torch distribution, you can use the `log_prob`and `sample` functions.

In [ ]:
# you can compute logprob and sample from your distribution:
print(prior.log_prob(torch.Tensor([0,0])))
print(prior.sample((3,)))

Below we define the core class of our RealNVP model. The `__init__` function defines the networks layers, the mask and the prior distribution that are stored in the `self` object, shared by all functions in the class.

> **Q5.** Complete the function `f` to compute the forward propagation function $f(\mathbf{x})$ and $\log \mathrm{det} J$.

> **Q6.** Complete the `log_prob` function to return the log probability.

> **Q7.** Complete the inverse function `g` returning $g(\mathbf{z}) = \mathbf{x}$.

> **Q8.** Complete the `sample` function to draw new samples from the flow.

In [ ]:
class RealNVP(nn.Module):
    def __init__(self, nets, nett, mask, prior):
        super(RealNVP, self).__init__()
        # Create a flow
        # nets:  a function that returns a PyTorch neural network, e.g., nn.Sequential, s = nets(), s: dim(X) -> dim(X)
        # nett:  a function that returns a PyTorch neural network, e.g., nn.Sequential, t = nett(), t: dim(X) -> dim(X)
        # mask:  a torch.Tensor of size #number_of_coupling_layers x #dim(X)
        # prior: an object from torch.distributions e.g., torch.distributions.MultivariateNormal
        self.prior = prior
        self.mask = mask
        self.t = torch.nn.ModuleList([nett() for _ in range(len(mask))])
        self.s = torch.nn.ModuleList([nets() for _ in range(len(mask))])

    def f(self, x):        
        # Compute f(x) = z and log_det_Jacobian of f, 
        #    where self.mask[i], self.t[i], self.s[i] define a i-th masked coupling layer   
        # x: a torch.Tensor, of shape batchSize x dim(X), is a datapoint
        # return z: a torch.Tensor of shape batchSize x dim(X), a hidden representations
        # return log_det_J: a torch.Tensor of len batchSize
        
        # YOUR CODE
        return z, log_det_J
    
    def log_prob(self, x):
        # Compute and return log p(x)
        # using the change of variable formula and log_det_J computed by f
        # return logp: torch.Tensor of len batchSize
        
        # YOUR CODE
        return logp
        
    def g(self, z):
        # Compute and return g(z) = x, 
        #    where self.mask[i], self.t[i], self.s[i] define a i-th masked coupling layer   
        # z: a torch.Tensor of shape batchSize x dim(X)
        # return x: a torch.Tensor of shape batchSize x dim(X)
        
        x = z
        # YOUR CODE
        return x
    
    def sample(self, batchSize): 
        # Draw and return batchSize samples from flow using implementation of g
        # return x: torch.Tensor of shape batchSize x dim(X)
        
        # YOUR CODE
        return x

Create an instance of the model with our implementation of `nets`, `nett`, `masks`, and `prior`.

In [ ]:
flow = RealNVP(nets, nett,  masks, prior)

> **Q9.** Check that the flow is indeed invertible such that $g(f(\mathbf{x}))=\mathbf{x}$. Hint: use the [`torch.allclose`](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html) function.

In [ ]:
# Check that a flow is invertible g(f(x)) = x Hint: torch.allclose
x = torch.randn((10,2))
# YOUR CODE

So far, our model is implemented but the parameters of the flow $\theta$ are randomly initialized. We need to train it using an optimizers and by minimizing the negative log-likelihood.

> **Q10.** Define an optimizer with $\eta = 10^{-4}$.

In [ ]:
optimizer =  # choose an optimizer, use module torch.optim

> **Q11.** Complete the training loop by generating $B=100$ and minimizing the loss. Recall the steps to follow to optimize a model in pyTorch. Comment on the loss evolution.

In [ ]:
n_steps = 5000  # Number of total steps to update the parameters
B = 100         # Number of data used each step

pbar = tqdm(range(n_steps))
loss_list = []
for t in pbar:
    # YOUR CODE to generate data, compute the loss and optimize the flow parameters
    
    pbar.set_description(f"loss: {loss.item():.3f}")

# Visualization

Draw several plots: 
- samples from flow
- samples from prior
- data samples
- mapping form data to prior

In [ ]:
noisy_moons = datasets.make_moons(n_samples=1000, noise=noise)[0].astype(np.float32)
z = flow.f(torch.from_numpy(noisy_moons))[0].detach().numpy()

plt.figure(figsize=(10, 10))
plt.subplot(221)
plt.scatter(z[:, 0], z[:, 1])
plt.title(r'$z = f(X)$')

z = prior.sample((1000,))
plt.subplot(222)
plt.scatter(z[:, 0], z[:, 1])
plt.title(r'$z \sim p(z)$')

plt.subplot(223)
x = datasets.make_moons(n_samples=1000, noise=.05)[0].astype(np.float32)
plt.scatter(x[:, 0], x[:, 1], c='r')
plt.title(r'$X \sim p(X)$')

plt.subplot(224)
x = flow.sample(1000).detach().numpy()
plt.scatter(x[:, 0], x[:, 1], c='r')
plt.title(r'$X = g(z)$')

Draw the estimated density:

In [ ]:
xpoints = np.linspace(-1.5, 2.5, 500)
ypoints = np.linspace(-1.0, 1.5, 500)
(x1, x2,) = np.meshgrid(xpoints, ypoints)
xgrid = np.concatenate((x1.reshape(-1, 1), x2.reshape(-1, 1)), axis=1).astype(np.float32)
p = np.exp(flow.log_prob(torch.from_numpy(xgrid)).detach().numpy())

In [ ]:
fig = plt.figure()
plt.imshow(
    p.reshape(x1.shape), aspect="equal", origin="lower")
plt.axis('off')
plt.show()